# Scraping Data Review Goodreads

Notebook ini digunakan untuk melakukan proses pengumpulan data review buku dari website Goodreads menggunakan teknik web scraping berbasis Selenium. Hasil scraping kemudian disimpan sebagai dataset mentah untuk tahap preprocessing dan modeling selanjutnya.


**Output:**
review_id_mentah.csv

File ini merupakan hasil scraping langsung dari Goodreads yang berisi kumpulan review buku.


**Kolom yang dihasilkan:**

- **review_id**  
  Identitas unik setiap review yang diambil dari halaman Goodreads.

- **review_text**  
  Isi teks review asli yang diperoleh dari website (belum melalui proses pembersihan atau preprocessing).

- **spoiler_label**  
  Label yang menunjukkan apakah review mengandung spoiler (1) atau tidak (0), berdasarkan informasi dari dataset atau hasil labeling awal.


Tahapan proses scraping:

1. Setup Environment  
   Menyiapkan library dan konfigurasi yang dibutuhkan untuk proses scraping.

2. Daftar URL Buku  
   Mengumpulkan daftar link halaman buku Goodreads yang akan di-scrape.

3. Ambil Resource via Selenium  
   Mengakses halaman web secara otomatis menggunakan Selenium untuk mengambil konten review.

4. Helper Function  
   Membuat fungsi bantu untuk mengekstrak data review dari halaman web.

5. Pipeline Scraping  
   Menjalankan proses scraping secara terstruktur dari awal hingga akhir untuk semua URL.

6. Simpan ke CSV  
   Menyimpan hasil scraping ke dalam file review_id_mentah.csv.

7. Statistik Dataset  
   Melakukan analisis awal seperti jumlah data dan distribusi review yang berhasil dikumpulkan.

## 1. Setup Environtment

In [1]:
# Install semua library yang dibutuhkan:
# selenium & selenium-wire: otomasi browser + intercept jaringan
# webdriver-manager: auto-download ChromeDriver yang cocok
# blinker==1.7.0: dependency selenium-wire (versi dikunci agar kompatibel)
# langdetect: deteksi bahasa teks review
# pandas, tqdm: manipulasi data & progress bar
# beautifulsoup4, requests: parsing HTML & HTTP request langsung ke API
!pip install selenium webdriver-manager selenium-wire blinker==1.7.0 langdetect pandas tqdm beautifulsoup4 requests

## b. Import & Konfigurasi

In [2]:
import time                                         # delay antar request agar tidak terlalu cepat
import random                                       # untuk generate delay acak
import re                                           # regex untuk bersihkan teks
import json                                         # parsing body request yang diintercept
import requests                                     # HTTP request langsung ke API (bypass selenium-wire)
import pandas as pd                                 # manipulasi data & simpan ke CSV
from tqdm.notebook import tqdm                      # progress bar di Jupyter
from langdetect import detect, LangDetectException  # deteksi bahasa teks review
from bs4 import BeautifulSoup                       # parsing HTML dari teks review

# import selenium & webdriver untuk otomasi browser
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from seleniumwire import webdriver as sw_driver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

# buat session requests terpisah yang bypass proxy bawaan selenium-wire,
# sehingga panggilan API GraphQL langsung ke server tanpa overhead proxy.
api_session = requests.Session()
api_session.trust_env = False

# konfigurasi scraping
OUTPUT_CSV     = "review_id_mentah.csv"
LIMIT_PER_PAGE = 30     # jumlah review per request ke API
MAX_PAGES      = None   # None = ambil semua halaman, int = batas halaman per buku
REQUEST_PAUSE  = (1, 3) # jeda acak antar request (detik), buat meminimalisir rate-limit

# endpoint & kredensial GraphQL
# URL & API key ini ditemukan dengan mengintercept traffic browser via selenium-wire saat halaman review Goodreads pertama kali dibuka
GRAPHQL_URL = "https://kxbwmqov6jgg3daaamb744ycu4.appsync-api.us-east-1.amazonaws.com/graphql"
API_KEY     = "da2-xpgsdydkbregjhpr6ejzqdhuwy"

# header HTTP yang dipakai setiap request GraphQL, meniru browser asli agar tidak diblokir oleh server Goodreads.
GRAPHQL_HEADERS = {
    "x-api-key"   : API_KEY,
    "content-type": "application/json",
    "accept"      : "application/graphql-response+json,application/json;q=0.9",
    "origin"      : "https://www.goodreads.com",
    "referer"     : "https://www.goodreads.com/",
    "user-agent"  : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
}

# query GraphQL untuk mengambil review per buku
# field yang diambil: id, teks, status spoiler, rating, waktu, dan info kreator
# pageInfo.nextPageToken dipakai untuk paginasi halaman berikutnya
GRAPHQL_QUERY = """
query getReviews($filters: BookReviewsFilterInput!, $pagination: PaginationInput) {
  getReviews(filters: $filters, pagination: $pagination) {
    totalCount
    edges {
      node {
        id
        text
        spoilerStatus
        rating
        createdAt
        updatedAt
        creator {
          name
          webUrl
        }
      }
    }
    pageInfo {
      nextPageToken
    }
  }
}
"""

## 2. Daftar URL Buku 
Daftar ini berisi kumpulan link halaman buku dari Goodreads yang akan digunakan sebagai sumber data utama untuk proses scraping review. Setiap URL merepresentasikan satu buku yang akan diambil review-nya secara otomatis menggunakan Selenium.

Sebelum digunakan, dilakukan proses filtering untuk memastikan semua link valid (dimulai dengan http), sehingga hanya URL yang benar-benar aktif yang diproses dalam pipeline scraping.

Jumlah total buku yang akan di-scrape kemudian ditampilkan untuk memastikan seluruh dataset sumber sudah terdefinisi dengan benar sebelum proses pengambilan data dimulai.

In [3]:
# daftar URL halaman buku Goodreads yang akan di-scrape reviewnya
# Setiap URL mengarah ke satu halaman buku, dan scraper akan mengambil semua review berbahasa Indonesia dari masing-masing buku
goodreads_links = [
    "https://www.goodreads.com/book/show/133227651-angsa-dan-kelelawar",                    
    "https://www.goodreads.com/book/show/25924054-the-dead-returns",                       
    "https://www.goodreads.com/book/show/123850391-teka-teki-rumah-aneh",                  
    "https://www.goodreads.com/en/book/show/205345677-pasien",                              
    "https://www.goodreads.com/book/show/20613611-malice",                                  
    "https://www.goodreads.com/book/show/1321926.The_Tokyo_Zodiac_Murders",                
    "https://www.goodreads.com/book/show/52699713-second-sister",                           
    "https://www.goodreads.com/en/book/show/36804340-the-good-son",                         
    "https://www.goodreads.com/en/book/show/22432940-girls-in-the-dark",                   
    "https://www.goodreads.com/book/show/32450412-holy-mother",                             
    "https://www.goodreads.com/book/show/204903103-tragedi-pedang-keadilan",                
    "https://www.goodreads.com/en/book/show/23847971-a-midsummer-s-equation",               
    "https://www.goodreads.com/book/show/19161835-confessions",                             
    "https://www.goodreads.com/book/show/2016000.Rahasia_Meede",                           
    "https://goodreads.com/book/show/50078136-dua-dini-hari",                             
    "https://www.goodreads.com/book/show/25082874-misteri-patung-garam",                   
    "https://www.goodreads.com/book/show/34300076-24-jam-bersama-gaspar",                   
    "https://www.goodreads.com/book/show/15990354-omen",                                    
    "https://www.goodreads.com/book/show/18000845-tujuh-lukisan-horor",
    "https://www.goodreads.com/book/show/18493770-misteri-organisasi-rahasia-the-judges",
    "https://www.goodreads.com/book/show/20581395-malam-karnaval-berdarah",
    "https://www.goodreads.com/book/show/22325046-kutukan-hantu-opera",
    "https://www.goodreads.com/book/show/23261061-sang-pengkhianat",
    "https://www.goodreads.com/book/show/25967996-target-terakhir",
    "https://www.goodreads.com/book/show/9475091-obsesi",
    "https://www.goodreads.com/book/show/11355995-pengurus-mos-harus-mati",
    "https://www.goodreads.com/book/show/12955672-permainan-maut",
    "https://www.goodreads.com/book/show/13637705-teror",
    "https://www.goodreads.com/en/book/show/57369066-burning-heat",
    "https://www.goodreads.com/book/show/53415555-penance",
    "https://www.goodreads.com/book/show/22432940-girls-in-the-dark",
    "https://www.goodreads.com/id/book/show/59842337-black-showman-dan-pembunuhan-di-kota-tak-bernama",
    "https://www.goodreads.com/book/show/43321712-murder-in-the-crooked-house",
    "https://www.goodreads.com/book/show/33602102-newcomer",
    "https://www.goodreads.com/book/show/238142982-about-a-place-in-the-kinki-region",
    "https://www.goodreads.com/book/show/222377017-guilt",
    "https://www.goodreads.com/book/show/30119105-the-borrowed",
    "https://www.goodreads.com/book/show/13419386-danur",
    "https://www.goodreads.com/book/show/3244500-rumah-lebah",
    "https://www.goodreads.com/book/show/32450412-holy-mother",
    "https://www.goodreads.com/book/show/43502793-ve",
    "https://www.goodreads.com/book/show/51228202-playing-victim",
]

# filter hanya URL yang valid (dimulai dengan 'http') untuk menghindari error saat request
goodreads_links = [url for url in goodreads_links if url.startswith("http")]
print(f"Total buku: {len(goodreads_links)}")

Total buku: 42


## 3. Ambil resourceId via Selenium

Selenium dipakai hanya sekali per buku untuk mengambil resourceId (format kca://work/...) yang dibutuhkan sebagai filter GraphQL.

Caranya dengan intercept request GraphQL pertama yang otomatis dipanggil saat halaman review dibuka.

In [4]:
# function untuk membuat instance Chrome yang dikelola selenium-wire
# selenium-wire memungkinkan intercept traffic jaringan untuk menangkap request GraphQL yang dikirim halaman Goodreads.
def build_sw_driver():
    # mode headless & stabilitas
    options = Options()
    options.add_argument("--headless=new")          # jalankan Chrome tanpa tampilan GUI
    options.add_argument("--no-sandbox")            # wajib di environment non-root (Colab, server)
    options.add_argument("--disable-dev-shm-usage") # cegah crash karena /dev/shm terbatas
    options.add_argument("--window-size=1440,900")  # ukuran viewport desktop agar layout tidak berubah

    # anti-deteksi bot
    # goodreads bisa memblokir browser jika terdeteksi sebagai automation
    # dua opsi ini menyembunyikan tanda-tanda bahwa Chrome dijalankan oleh Selenium
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) " # hapus flag 'Chrome is controlled by automated software'
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36" # nonaktifkan ekstensi bawaan automation
    )

    # ChromeDriverManager otomatis mendownload ChromeDriver yang sesuai versi Chrome
    options.add_experimental_option("excludeSwitches", ["enable-automation"])  
    options.add_experimental_option("useAutomationExtension", False)          
    service = Service(ChromeDriverManager().install())
    return sw_driver.Chrome(service=service, options=options)

## 4. Helper Functions
Helper functions ini berisi kumpulan fungsi pendukung yang digunakan selama proses scraping data review dari Goodreads. 

Fungsi-fungsi tersebut mencakup pengecekan bahasa untuk memastikan hanya review berbahasa Indonesia yang digunakan, pembersihan teks dari HTML dan karakter tidak relevan, pengambilan data review dari GraphQL API dengan sistem pagination dan retry untuk menjaga kestabilan request, serta ekstraksi resourceId menggunakan Selenium melalui intercept request halaman review. 

Seluruh fungsi ini bekerja sebagai pipeline pendukung utama untuk memastikan data yang diambil bersih, valid, dan dapat digunakan untuk tahap analisis berikutnya.

In [5]:
# function untuk mendeteksi apakah teks review berbahasa Indonesia
# return True jika teks terdeteksi sebagai bahasa Indonesia
def is_indonesian(text, min_chars=30):
    # teks terlalu pendek tidak bisa dideteksi bahasanya secara akurat, langsung skip
    if not text or len(text.strip()) < min_chars:
        return False
    try:
        return detect(text) == "id" # id itu kode bahasa Indonesia
    except LangDetectException:
        return False # langdetect lempar exception jika teks ambigu, anggap bukan Indonesia


# function untuk membersihkan teks review dari tag HTML dan whitespace berlebih 
def clean_text(html_text):
    # teks kosong atau terlalu pendek langsung dikembalikan sebagai string kosong
    if not html_text or len(html_text.strip()) < 3:
        return ""
    
    # hapus tag <spoiler>...</spoiler> yang kadang muncul di teks review Goodreads
    text = re.sub(r"</?spoiler>", "", html_text, flags=re.I)

    # parse HTML hanya jika teks masih mengandung tag HTML ('<'), hindari overhead BeautifulSoup
    if "<" in text:
        text = BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)

    # normalisasi whitespace, ganti spasi/tab/newline berlebih menjadi satu spasi
    return re.sub(r"\s+", " ", text).strip()


# function untuk fetch satu halaman review dari GraphQL API 
# mendukung paginasi via cursor (nextPageToken) dan retry otomatis hingga 3x
# return (edges, next_cursor, total_count) atau (None, None, None) jika gagal.
def fetch_reviews_page(resource_id, cursor=None, limit=30):
    # susun parameter paginasi, tambahkan cursor hanya jika bukan halaman pertama
    pagination = {"limit": limit}
    if cursor:
        pagination["after"] = cursor # cursor berisi nextPageToken dari respons sebelumnya

    # payload dikirim sebagai JSON body ke endpoint GraphQL,
    # formatnya meniru request yang dikirim browser Goodreads asli
    payload = {
        "operationName": "getReviews",
        "variables": {
            "filters": {
                "resourceType": "WORK",
                "resourceId": resource_id # ID unik buku yang didapat dari intercept request halaman review
            },
            "pagination": pagination
        },
        "extensions": {"clientLibrary": {"name": "@apollo/client", "version": "4.1.6"}},
        "query": GRAPHQL_QUERY
    }

    # coba kirim request hingga 3x; jika gagal, tunggu 5 detik sebelum retry berikutnya
    for attempt in range(3): 
        try:
            # kirim request, parse JSON, lalu ambil field getReviews dari respons
            resp = api_session.post(GRAPHQL_URL, headers=GRAPHQL_HEADERS, json=payload, timeout=30)
            data = resp.json()
            result = data.get("data", {}).get("getReviews")
            # jika API mengembalikan error (misalnya resource tidak ditemukan) return None, None, None
            if result is None:
                print(f"  [API ERR] {data.get('errors', 'unknown')}")
                return None, None, None
            # Kembalikan: daftar review, token halaman berikutnya, dan total review tersedia
            return result["edges"], result["pageInfo"]["nextPageToken"], result["totalCount"]
        except Exception as e:
            print(f"  [REQ ERR attempt {attempt+1}] {e}")
            time.sleep(5) # tunggu sebelum retry agar tidak langsung gagal lagi

    return None, None, None # semua retry habis, kembalikan None

# function untuk mendapatkan resourceId dari halaman review Goodreads
# resourceId ini diperlukan untuk query GraphQL, dan biasanya tersembunyi di dalam request jaringan yang dikirim halaman review
# prosesnya: Buka halaman buku -> navigasi ke halaman review -> intercept request GraphQL
# return: resourceId string atau None jika gagal.
def get_resource_id(driver, book_url):
    # nama buku dari slug URL, untuk log
    book_name = book_url.split("/")[-1].replace("-", " ").title()

    # bersihkan riwayat request lama lalu buka halaman buku, tunggu 3 detik agar halaman selesai render
    driver.requests.clear()
    driver.get(book_url)
    time.sleep(3)

    try:
        # cari dan klik link ke halaman reviews, tunggu max 15 detik
        btn = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//a[contains(@href, '/reviews')]"
            ))
        )
        reviews_url = btn.get_attribute("href")

        # validasi, pastikan link mengarah ke Goodreads, bukan link eksternal
        if "goodreads.com" not in reviews_url:
            print(f"[{book_name}] URL bukan Goodreads reviews: {reviews_url}")
            return None

    except TimeoutException:
        print(f"[{book_name}] Tombol reviews tidak ditemukan")
        return None

    # bersihkan request lama lagi, lalu buka halaman reviews
    # (halaman ini akan otomatis memicu request GraphQL yang kita butuhkan)
    driver.requests.clear()
    driver.get(reviews_url)
    time.sleep(5)

    # telusuri semua request yang ditangkap selenium-wire, cari yang ke endpoint AppSync
    # cari yang menuju endpoint AppSync (GraphQL Goodreads) dan punya body
    for req in driver.requests:
        # "appsync" memastikan hanya request ke GraphQL Goodreads yang diproses,
        # req.body memastikan request punya isi (bukan GET request atau request kosong)
        if "appsync" in req.url and req.body:
            try:
                # decode bytes -> string -> dict
                body = json.loads(req.body.decode("utf-8"))
                # pastikan ini request getReviews, bukan operasi GraphQL lain di halaman yang sama
                if body.get("operationName") == "getReviews": 
                    resource_id = body["variables"]["filters"]["resourceId"]
                    print(f"[{book_name}] resourceId: {resource_id}")
                    return resource_id
            except Exception:
                continue # body tidak bisa di-decode atau bukan JSON, coba request berikutnya

    print(f"[{book_name}] Tidak bisa ekstrak resourceId")
    return None

## 5. Pipeline Utama
Pipeline ini digunakan untuk mengumpulkan data review dari Goodreads secara bertahap dan terstruktur. 

Proses dimulai dengan mengambil resourceId dari setiap buku menggunakan Selenium dan menyimpannya ke file JSON agar dapat digunakan ulang tanpa perlu scraping dari awal. Setelah itu, data review diambil langsung melalui GraphQL API berdasarkan resourceId, kemudian setiap review dibersihkan, difilter hanya untuk bahasa Indonesia, diberi label spoiler, dan dihindari dari duplikasi. Seluruh hasil scraping disimpan secara berkala ke file CSV sebagai checkpoint hingga semua buku selesai diproses dan dataset akhir berhasil terbentuk.

In [7]:
# ambil semua resourceId dulu, simpan ke JSON 
import json
import os

RESOURCE_MAP_FILE = "resource_map.json" # file cache agar resourceId tidak perlu diambil ulang

# langkah 1: ambil resourceId untuk setiap buku
# kalau file cache sudah ada dari run sebelumnya, langsung load isi file sebagai dict {book_url: resource_id}
if os.path.exists(RESOURCE_MAP_FILE):
    with open(RESOURCE_MAP_FILE) as f:
        resource_map = json.load(f)
    print(f"Resource map loaded: {len(resource_map)} buku") # format {book_url: resource_id}
# kalau belum ada, jalankan Selenium untuk setiap buku dan simpan hasilnya
else:
    resource_map = {}
    # iterasi semua url buku
    for book_url in tqdm(goodreads_links, desc="Ambil resourceId"):
        # skip jika sudah ada
        if book_url in resource_map:
            continue
        # ambil nama buku dari slug url untuk log
        book_name = book_url.split("/")[-1].replace("-", " ").title()

        # buat driver baru per buku agar tidak ada state/session yang bocor antar buku
        driver = build_sw_driver()
        # buka halaman buku dan intercept resourceId
        resource_id = get_resource_id(driver, book_url)
        # tutup browser
        driver.quit() 

        # jika berhasil dapat resourceId
        if resource_id:
            resource_map[book_url] = resource_id # simpan ke dict dengan url sebagai key
            print(f"OK: {book_name}")
        else:
            print(f"GAGAL: {book_url}")
        time.sleep(2)

    # simpan ke file JSON sebagai cache permanen
    with open(RESOURCE_MAP_FILE, "w") as f:
        json.dump(resource_map, f, indent=2)
    print(f"Tersimpan: {len(resource_map)} resourceId")

# langkah 2: scraping review via GraphQL API (tanpa Selenium) 
all_reviews = []
seen_ids = set()

# iterasi tiap buku
for book_url, resource_id in tqdm(resource_map.items(), desc="Scraping"):
    print(f"\nMemproses: {book_url}")

    cursor = None   # cursor paginasi, None = mulai dari halaman pertama
    page = 0
    book_count = 0  # counter review Indonesia yang berhasil dikumpulkan untuk buku ini

    # loop paginasi, terus ambil halaman berikutnya sampai tidak ada cursor lagi
    while True:
        # hentikan jika sudah mencapai batas halaman yang ditentukan
        if MAX_PAGES is not None and page >= MAX_PAGES:
            print(f"Batas halaman ({MAX_PAGES}) tercapai.")
            break

        # fetch satu halaman review
        edges, next_cursor, total = fetch_reviews_page(resource_id, cursor, LIMIT_PER_PAGE)

        # request gagal bahkan setelah retry
        if edges is None: 
            print("Gagal fetch, hentikan buku ini.")
            break

        # hanya tampilkan total review sekali di awal tiap buku
        if page == 0:
            print(f"Total review di Goodreads: {total}") # tampilkan total hanya sekali di awal

        new_this_page = 0

        # iterasi tiap review di halaman ini
        for edge in edges:
            # data review ada di dalam key "node"
            node = edge["node"] 
            # node bisa kosong jika review sudah dihapus di goodreads
            if node is None:
                continue

            # id unik review
            review_id = node.get("id", "")
            # skip duplikat
            if review_id in seen_ids:
                continue  

            # bersihkan html dari teks review lalu cek apakah bahasa Indonesia
            clean = clean_text(node.get("text") or "")
            # buang review non-Indonesia
            if not is_indonesian(clean):
                continue
            spoiler_label = 1 if node.get("spoilerStatus") else 0 # konversi boolean ke int

            # tandai sudah diproses
            seen_ids.add(review_id)
            all_reviews.append({
                "book_url"     : book_url,
                "review_id"    : review_id,
                "reviewer"     : node["creator"]["name"] if node.get("creator") else "",
                "reviewer_url" : node["creator"]["webUrl"] if node.get("creator") else "",
                "rating"       : node.get("rating"),
                "created_at"   : node.get("createdAt"),
                "review_text"  : clean,
                "spoiler_label": spoiler_label,
            })
            new_this_page += 1 # tambah counter review baru di halaman ini
            book_count += 1 # tambah counter total review untuk buku ini

        page += 1 # increment nomor halaman setelah selesai diproses
        print(f"Halaman {page}: {len(edges)} dari API, {new_this_page} review bahasa Indonesia baru (total review bahasa Indonesia: {book_count})")

        # tidak ada halaman berikutnya, selesai untuk buku ini
        if not next_cursor:
            print("Selesai.")
            break

        # increment nomor halaman setelah selesai diproses
        cursor = next_cursor
        time.sleep(random.uniform(*REQUEST_PAUSE)) # jeda acak antar request untuk menghindari rate limit

    # checkpoint: simpan progres setiap selesai 1 buku agar data tidak hilang jika scraping gagal di tengah jalan
    df_temp = pd.DataFrame(all_reviews)
    df_temp.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Checkpoint disimpan: {len(df_temp)} review")

print(f"\nTotal review Indonesia terkumpul: {len(all_reviews)}")

Ambil resourceId:   0%|          | 0/42 [00:00<?, ?it/s]

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=64797): Read timed out. (read timeout=120)

## 6. Simpan ke CSV

In [ ]:
# konversi list hasil scraping menjadi DataFrame
df = pd.DataFrame(all_reviews)
if not df.empty:
    # simpan ke CSV dengan encoding utf-8-sig agar karakter Indonesia terbaca benar di Excel
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Disimpan: {len(df)} review")
    print(f"Spoiler: {df['spoiler_label'].sum()}")
    print(f"Non-spoiler: {(df['spoiler_label']==0).sum()}")
    print(f"Dari berapa buku: {df['book_url'].nunique()}")
else:
    print("Tidak ada data.")

## 7. Statistik Dataset
Untuk melihat gambaran awal dari dataset hasil scraping sebelum masuk ke tahap preprocessing dan modeling. Mencakup:
- Distribusi label spoiler dan non-spoiler untuk melihat keseimbangan data
- Distribusi jumlah review pada setiap buku untuk mengetahui sebaran data per sumber
- Analisis panjang teks review untuk memahami karakteristik umum data seperti rata-rata panjang dan variasinya. 

In [ ]:
# membuat DataFrame dari seluruh review yang telah dikumpulkan selama proses scraping
if not df.empty:
    print("Distribusi Label")
    print(df["spoiler_label"].value_counts().rename({0: "Non-Spoiler", 1: "Spoiler"}))
    print()
    print("Distribusi per Buku")
    print(
        df.groupby("book_url")["spoiler_label"]
          .agg(total="count", spoiler="sum")
          .assign(pct_spoiler=lambda x: (x["spoiler"] / x["total"] * 100).round(1))
    )
    print()
    print("Panjang Teks Review")
    df["text_len"] = df["review_text"].str.len()
    print(df["text_len"].describe().round(1))